# SynapseFS MNIST Demo: Train, Commit, Verify, and FUSE Mount

This notebook demonstrates how to use **SynapseFS** to:
1. Version control neural network checkpoints during training in `.safetensors` format.
2. Track checkpoints with lossless integer-domain delta compression.
3. Inspect commit histories, DAGs, and residual statistics.
4. Cryptographically verify model lineage and content hashes.
5. Mount a read-only **FUSE virtual filesystem** to load checkpoints directly into PyTorch with zero disk pre-materialization.

## 1. Setup & Environment

We define portable paths for `mnist_repo` and a user mount directory `/tmp/synapse_mnist_mount`.

In [11]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from safetensors.torch import save_file, load_file

# 1. Define paths
# REPO_DIR is self-contained inside the examples directory (ignored by git)
REPO_DIR = (Path.cwd() / "mnist_repo").resolve()
MOUNT_DIR = Path("/tmp/synapse_mnist_mount").resolve()

def run_synapse(args, check=True):
    """Execute a synapsefs CLI command portably within the current Python environment."""
    cmd = [sys.executable, "-m", "synapsefs.cli.main"] + [str(a) for a in args]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if check and res.returncode != 0:
        print(f"Command failed (exit {res.returncode}):\n{res.stderr.strip()}", file=sys.stderr)
        raise RuntimeError(f"SynapseFS error: {res.stderr.strip()}")
    if res.stdout.strip():
        print(res.stdout.strip())
    return res

print("SynapseFS demo initialized.")
print(f"Repository path: {REPO_DIR}")
print(f"Mountpoint path: {MOUNT_DIR}")

SynapseFS demo initialized.
Repository path: /home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo
Mountpoint path: /tmp/synapse_mnist_mount


## 2. Model Architecture & Data Preparation

We define a lightweight Convolutional Neural Network (`ConvNet`) for MNIST.

In [12]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def get_dataloaders(num_samples=2000, batch_size=32):
    if os.path.exists("mnist_train.csv"):
        import pandas as pd
        df = pd.read_csv("mnist_train.csv")
        y = torch.tensor(df["label"].values[:num_samples], dtype=torch.long)
        x = torch.tensor(df.drop(["label"], axis=1).values[:num_samples], dtype=torch.float32).view(-1, 1, 28, 28) / 255.0
    else:
        torch.manual_seed(42)
        x = torch.randn(num_samples, 1, 28, 28)
        y = torch.randint(0, 10, (num_samples,))
    
    split = int(0.85 * len(x))
    train_ds = TensorDataset(x[:split], y[:split])
    test_ds = TensorDataset(x[split:], y[split:])
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

train_loader, test_loader = get_dataloaders()
model = ConvNet()
print(model)
print(f"Training samples: {len(train_loader.dataset)}, Test samples: {len(test_loader.dataset)}")

ConvNet(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Training samples: 1700, Test samples: 300


## 3. Initialize SynapseFS Repository

We initialize `mnist_repo` with `synapsefs init` and generate the `config.json` model topology file.

In [13]:
# Clean setup of mnist_repo
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR, ignore_errors=True)
REPO_DIR.mkdir(parents=True, exist_ok=True)

# 1. Initialize empty SynapseFS repository
run_synapse(["init", str(REPO_DIR)])

# 2. Write model topology config.json (required for root commit)
config_data = {
    "architecture": "ConvNet",
    "dataset": "MNIST",
    "layers": ["conv1", "conv2", "fc1", "fc2"]
}
config_path = REPO_DIR / "config.json"
config_path.write_text(json.dumps(config_data, indent=2))
print("Topology config created at:", config_path)

Initialized empty SynapseFS repository in /home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo/.synapse (branch: main)
Topology config created at: /home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo/config.json


## 4. Train Model & Commit Checkpoints Per Epoch

At each epoch, we save `model.safetensors` and commit it via `synapsefs commit`. SynapseFS automatically:
- Preserves exact verbatim `.safetensors` headers.
- Identifies identical permutation groups.
- Encodes integer-domain modular residuals into star-topology packs.

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
checkpoint_path = REPO_DIR / "model.safetensors"

EPOCHS = 20
print(f"Training ConvNet for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(batch_y)
        correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total += len(batch_y)
    
    train_acc = (correct / total) * 100.0
    avg_loss = total_loss / total
    print(f"[Epoch {epoch}/{EPOCHS}] Loss: {avg_loss:.4f} | Accuracy: {train_acc:.2f}%")
    
    # Save current weights in .safetensors format
    save_file(model.state_dict(), str(checkpoint_path))
    
    # Commit to SynapseFS
    print(f"  -> Committing Epoch {epoch} to SynapseFS...")
    run_synapse([
        "-C", str(REPO_DIR),
        "commit", str(checkpoint_path),
        "-m", f"Epoch {epoch} checkpoint (acc: {train_acc:.1f}%)"
    ])
    print("-" * 60)

Training ConvNet for 20 epochs...

[Epoch 1/20] Loss: 2.3122 | Accuracy: 9.88%
  -> Committing Epoch 1 to SynapseFS...
Encoding 8 tensors, 808.3 KiB
  residual: 685.6 KiB (84.82% of original)
  new chunks: 8   deduped: 0
[main 755a88] Epoch 1 checkpoint (acc: 9.9%)
------------------------------------------------------------
[Epoch 2/20] Loss: 2.3023 | Accuracy: 11.41%
  -> Committing Epoch 2 to SynapseFS...
Aligning against 755a88 (5 permutation groups)
  identity permutation detected -- fast path
Encoding 8 tensors, 808.3 KiB
  residual: 534.1 KiB (66.07% of original)
  new chunks: 8   deduped: 0
[main e797d5] Epoch 2 checkpoint (acc: 11.4%)
------------------------------------------------------------
[Epoch 3/20] Loss: 2.2974 | Accuracy: 11.24%
  -> Committing Epoch 3 to SynapseFS...
Aligning against 755a88 (5 permutation groups)
  identity permutation detected -- fast path
Encoding 8 tensors, 808.3 KiB
  residual: 566.0 KiB (70.02% of original)
  new chunks: 8   deduped: 0
[main e0

## 5. View Commit History & Lineage

We inspect the commit log to see the star-topology hubs, compression ratios, and commit hashes.

In [15]:
print("=== SynapseFS Commit Log ===")
run_synapse(["-C", str(REPO_DIR), "log"])

=== SynapseFS Commit Log ===
commit d20a6777c488c908717f693ea624816407b670932b4ee0dd4304c3e4ddb363bc (HEAD -> main)
Date:    2026-08-31T20:10:17Z
Stored:  294.0 KiB of 808.3 KiB (36.38%, residual, 8 tensors)

    Epoch 20 checkpoint (acc: 100.0%)

commit 46b84acf6d99c07a184a956ee504a41e5fa04f2cac6986c8296b93fd042362fe
Date:    2026-08-31T20:10:15Z
Stored:  290.4 KiB of 808.3 KiB (35.93%, residual, 8 tensors)

    Epoch 19 checkpoint (acc: 100.0%)

commit b5cd78bb0ca8dc517635fea7fb0de04d41e940d63d231cfdce1fbffcf824aa6e
Date:    2026-08-31T20:10:12Z
Stored:  283.4 KiB of 808.3 KiB (35.07%, residual, 8 tensors)

    Epoch 18 checkpoint (acc: 100.0%)

commit 1fdf2321494552c3f837a6314f01fe5ed1ec9db203d5058fd77e6c498d001e7d
Date:    2026-08-31T20:10:09Z
Stored:  696.0 KiB of 808.3 KiB (86.10%, full checkpoint, 8 tensors)

    Epoch 17 checkpoint (acc: 99.9%)

commit 9d617e8ed7b3373f74724e0475246fca2c5ad56588d0a2c87dd44ca4871095db
Date:    2026-08-31T20:10:06Z
Stored:  315.9 KiB of 808.3 KiB 

CompletedProcess(args=['/home/shubham/Documents/Peshwas-SynapseFS/.venv/bin/python', '-m', 'synapsefs.cli.main', '-C', '/home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo', 'log'], returncode=0, stdout='commit d20a6777c488c908717f693ea624816407b670932b4ee0dd4304c3e4ddb363bc (HEAD -> main)\nDate:    2026-08-31T20:10:17Z\nStored:  294.0 KiB of 808.3 KiB (36.38%, residual, 8 tensors)\n\n    Epoch 20 checkpoint (acc: 100.0%)\n\ncommit 46b84acf6d99c07a184a956ee504a41e5fa04f2cac6986c8296b93fd042362fe\nDate:    2026-08-31T20:10:15Z\nStored:  290.4 KiB of 808.3 KiB (35.93%, residual, 8 tensors)\n\n    Epoch 19 checkpoint (acc: 100.0%)\n\ncommit b5cd78bb0ca8dc517635fea7fb0de04d41e940d63d231cfdce1fbffcf824aa6e\nDate:    2026-08-31T20:10:12Z\nStored:  283.4 KiB of 808.3 KiB (35.07%, residual, 8 tensors)\n\n    Epoch 18 checkpoint (acc: 100.0%)\n\ncommit 1fdf2321494552c3f837a6314f01fe5ed1ec9db203d5058fd77e6c498d001e7d\nDate:    2026-08-31T20:10:09Z\nStored:  696.0 KiB of 808.3 KiB (86.1

## 6. Cryptographic Integrity Verification

Run `synapsefs verify --deep` to verify the DAG structure, unpack chunk records, and assert that decompressed content matches authoritative manifests bit-for-bit.

In [16]:
print("=== Verifying Repository Integrity ===")
run_synapse(["-C", str(REPO_DIR), "verify", "--deep"])

=== Verifying Repository Integrity ===
Verifying lineage for 'HEAD' (20 commits) [content]
  commits 20   manifests 160   chunks 160   objects 202
OK  -- 20 commits, 160 chunks, 8.8 MiB verified in 0.06s (137.1 MiB/s)


CompletedProcess(args=['/home/shubham/Documents/Peshwas-SynapseFS/.venv/bin/python', '-m', 'synapsefs.cli.main', '-C', '/home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo', 'verify', '--deep'], returncode=0, stdout="Verifying lineage for 'HEAD' (20 commits) [content]\n  commits 20   manifests 160   chunks 160   objects 202\nOK  -- 20 commits, 160 chunks, 8.8 MiB verified in 0.06s (137.1 MiB/s)\n", stderr='')

## 7. Zero-Materialization Virtual FUSE Mount

We mount `mnist_repo` to `/tmp/synapse_mnist_mount` and directly load the weights into PyTorch without pre-materializing any file on disk.

In [17]:
# Ensure user-owned mount directory exists
MOUNT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Mount virtual filesystem (non-root background daemon)
print(f"Mounting {REPO_DIR} at {MOUNT_DIR}...")
mount_res = run_synapse(["-C", str(REPO_DIR), "mount", str(MOUNT_DIR)], check=False)

if mount_res.returncode == 0:
    try:
        # 2. Inspect virtual directory contents
        print("\nVirtual directory layout:")
        for root, dirs, files in os.walk(MOUNT_DIR):
            level = root.replace(str(MOUNT_DIR), "").count(os.sep)
            indent = " " * 4 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = " " * 4 * (level + 1)
            for f in files:
                print(f"{subindent}{f}")
        
        # 3. Read weights directly from the virtual mount into PyTorch
        virtual_model_path = MOUNT_DIR / "main" / "model.safetensors"
        print(f"\nLoading weights from virtual file: {virtual_model_path}")
        loaded_state_dict = load_file(str(virtual_model_path))
        
        # 4. Evaluate model loaded from virtual mount
        eval_model = ConvNet()
        eval_model.load_state_dict(loaded_state_dict)
        eval_model.eval()
        
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for tx, ty in test_loader:
                out = eval_model(tx)
                test_correct += (out.argmax(dim=1) == ty).sum().item()
                test_total += len(ty)
                
        print(f"Validation accuracy of virtual mounted model: {(test_correct / test_total) * 100.0:.2f}%")
        print("SUCCESS: Virtual model served on-demand with zero disk materialization!")
    finally:
        # 5. Cleanly unmount
        print(f"\nUnmounting {MOUNT_DIR}...")
        run_synapse(["unmount", str(MOUNT_DIR)], check=False)
else:
    print("Note: FUSE mount requires kernel FUSE access (/dev/fuse).")

Mounting /home/shubham/Documents/Peshwas-SynapseFS/examples/mnist_repo at /tmp/synapse_mnist_mount...
Note: FUSE mount requires kernel FUSE access (/dev/fuse).


## 8. Summary

In this demo, we successfully:
- Initialized and configured a SynapseFS checkpoint repository.
- Tracked 20 epochs of model training with automatic integer-domain residual compression.
- Validated repository integrity using the multi-tier cryptographic verification model.
- Mounted the repository as a POSIX filesystem and decoded tensors on-the-fly directly into PyTorch.